# DRIVE Baseline — Attention Residual U-Net

**Target:** Vessel Dice ≥ 0.65 (DRIVE is the hardest of our 5 datasets — multi-blob vessel topology and only 20 labelled samples).

## Important caveats
- Only the **training** split has vessel annotations publicly. Test-set ground truth is hidden by the DRIVE challenge.
- We use the 20 labelled images and apply our own 15/3/2 patient-level split.
- Heavy augmentation in the YAML compensates for low sample count.
- Expect overfitting risk — early stop with `patience=20` watches for it.

## Kaggle setup
1. Add datasets: `andrewmvd/drive-digital-retinal-images-for-vessel-extraction` + `iteris-pkg`
2. Accelerator: GPU T4 x2
3. Persistence: Files only
4. Save Version → Save & Run All (Commit)

## 0 · Install (no kernel restart needed)

In [ ]:
import subprocess
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
REQ      = PKG_ROOT / 'requirements.txt'

subprocess.run(['pip', 'install', '-r', str(REQ), '--quiet', '--upgrade'], check=True)
print(f'✓ Setup complete. Package at: {PKG_ROOT}')

## 1 · Load config + locate DRIVE dataset

In [ ]:
import sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
PKG_ROOT = init_files[0].parent.parent
sys.path.insert(0, str(PKG_ROOT))

from iteris.config import load_config
from iteris.utils  import get_device

cfg = load_config(str(PKG_ROOT / 'configs' / 'drive.yaml'))

# Locate DRIVE root — auto-descend if Kaggle's wrapper is nested.
drive_candidates = [p for p in Path('/kaggle/input').rglob('DRIVE')
                    if p.is_dir() and (p / 'training' / '1st_manual').is_dir()]
if drive_candidates:
    cfg['data_root'] = str(drive_candidates[0].parent)
else:
    cfg['data_root'] = '/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction'
cfg['checkpoint_dir'] = '/kaggle/working'

print(f'Package root : {PKG_ROOT}')
print(f'Data root    : {cfg["data_root"]}')
print(f'Dataset      : {cfg["dataset"]}  ({cfg["modality"]})')
print(f'Image size   : {cfg["image_size"]}  in_channels={cfg["in_channels"]}')
print(f'Classes      : {cfg["num_classes"]} — {cfg["class_names"]}')
print(f'Normalisation: {cfg["normalize"]}   binarise labels: {cfg["binarize_labels"]}')
print(f'Epochs       : {cfg["epochs"]}  batch {cfg["batch_size"]}  lr {cfg["lr"]}  patience {cfg["patience"]}')
print(f'Device       : {get_device()}')

## 2 · Train

In [ ]:
from iteris.training import run_training

result      = run_training(cfg)
model       = result['model']
history     = result['history']
best_dice   = result['best_dice']
test_loader = result['test_loader']
checkpoint  = result['checkpoint']

print(f'\n✓ Training done. Best val Dice = {best_dice:.4f}  |  Checkpoint: {checkpoint}')

## 3 · Learning curves

In [ ]:
from iteris.visualization import plot_learning_curves
plot_learning_curves(history, cfg, target_dice=0.65)

## 4 · Test-set evaluation

In [ ]:
from iteris.evaluation import evaluate_test_set
scores_df = evaluate_test_set(model, test_loader, cfg)
print(scores_df.head())

## 5 · Export predicted masks (init state for DRL agents)

In [ ]:
from iteris.evaluation import export_predicted_masks
export_predicted_masks(model, test_loader, cfg)

## 6 · Qualitative grid

In [ ]:
from iteris.visualization import plot_qualitative_grid
plot_qualitative_grid(model, test_loader, cfg, n_show=4)

## 7 · Summary JSON

In [ ]:
from iteris.evaluation import save_summary_json
save_summary_json(history, scores_df, cfg, best_dice)